# PSI Calculations (removing small area and ignoring "Other" category)

* Prepare colonies dataset **[DONE]**
    * Import colonies file (identify which one) **[DONE]**
    * Add column `exclude_from_psi` **[DONE]**
    * All area_km2 < .0001 get `exclude_from_psi` = True **[DONE]**
    * All USO types that should be ignore get `exclude_from_psi` = True **[DONE]**
    * Calculate only bounding box neighbors **[DONE]**
    * Turn into a function that I can easily change **[DONE]**
* Removing `exclude_from_psi` from index calculations **[DONE]**
    * If the row has a USO category equal to one of the values in `remove_uso_category`, it assigns -1 to the value of the PCEN.
    * Make sure that final PSI ignores all excluded polygons in its calculations.
* Calculate index with PCEN divided by (1) Population; (2) Population/Area; and (3) 1.
    * Refactor code to have these (and other options).
    * Can I pass in a variable that says what the denominator should be? Even hard coding this in with Python command that executes code from a string?
* Calculate Average PSI for all Services **[DONE]**
* Calculate Normalized PSI for all Services, using min-max method. **[DONE]**
    * Combine the top two above into one function
    * Embed this function in the larger function `calc_all_services` or its equivalent

## Import modules and set constants

In [ ]:
import os
import pickle
from importlib import reload
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box
import spatial_index_utils

reload(spatial_index_utils)

# WGS 84 / Delhi
epsg_code = 7760

# Columns to remove for export to ESRI Shapefile
bbox_drop_columns = ['nbrs_bbox', 'nbrs_dist_bbox', 'centroid']

## Import colonies and do pre-processing

In [ ]:
from spatial_index_utils import generate_colonies_with_exclusions, calc_all_services

In [ ]:
colonies_pkl_file = 'colonies_bbox_nbrs25Aug2020.pkl'
columns_to_drop = ['nbrs_bbox', 'nbrs_dist_bbox', 'index']
uso_types_to_drop = ['Other']
area_cutoff_km2 = .0001
colonies = generate_colonies_with_exclusions(colonies_pkl_file = colonies_pkl_file,
                                             columns_to_drop = columns_to_drop, 
                                             uso_types_to_drop = uso_types_to_drop,
                                             area_cutoff_km2 = area_cutoff_km2)

In [ ]:
colonies_bbox_nbrs = colonies.copy()

## Import services shapefiles

In [ ]:
# Define filepaths

services_dir = os.path.join('shapefiles', 'Spatial_Index_GIS', 'Public Services')

bank_fp = os.path.join(services_dir, 'Banking', 'Banking.shp')
health_fp = os.path.join(services_dir, 'Health', 'Health.shp')
road_fp = os.path.join(services_dir, 'Major Road', 'Road.shp')
police_fp = os.path.join(services_dir, 'Police', 'Police Station.shp')
ration_fp = os.path.join(services_dir, 'Ration', 'Ration.shp')
school_fp = os.path.join(services_dir, 'School', 'schools7760.shp')
transport_fp = os.path.join(services_dir, 'Transport', 'Transport.shp')

# boundary of Delhi
delhi_bounds_filepath = os.path.join('shapefiles', 'delhi_bounds_buffer.shp')

# Check that all filepaths exist
filepath_list = [bank_fp, health_fp, road_fp, police_fp, ration_fp, school_fp, transport_fp, delhi_bounds_filepath]

for filepath in filepath_list:
    if not os.path.exists(filepath):
        print('{} does not exist'.format(filepath))

In [ ]:
# Import services
bank = gpd.read_file(bank_fp)
health = gpd.read_file(health_fp)
road = gpd.read_file(road_fp)
police = gpd.read_file(police_fp)
ration = gpd.read_file(ration_fp)
school = gpd.read_file(school_fp)
transport = gpd.read_file(transport_fp)

No need to check validity of these shapefiles, as this was previously done.

In [ ]:
bank.crs == health.crs == road.crs == police.crs == ration.crs == school.crs == transport.crs == colonies_bbox_nbrs.crs

In [ ]:
# Define all point services as dictionary
# makes it easier to calculate all point
# services with one function
point_services = {'bank': bank,
                  'health': health,
                  'police': police,
                  'ration': ration,
                  'school': school,
                  'transport': transport}

line_services = {'road': road}

### Calculate PSI for bbox neighbors using Population Size

In [ ]:
colonies_bbox_psi_popsize = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       pcen_denom = 'pop',
                                       nbr_dist_colname = 'nbrs_dist_bbox')

In [ ]:
colonies_bbox_psi_popsize.head()

In [ ]:
def export_shapefile(gdf, filename, columns_to_drop):
    """Save as ESRI Shapefile, Pickle object, and CSV"""
        
    shapefile = filename+'.shp'
    csv_file = filename+'.csv'
    pickle_file = filename+'.pkl'
    
    # Save as ESRI Shapefile
    gdf.drop(columns=columns_to_drop).to_file(shapefile)    
    
    # Save as CSV file
    gdf.to_csv(csv_file)
    
    # Save as Pickle file
    with open(pickle_file, 'wb') as f:
        pickle.dump(gdf, f)

In [ ]:
export_shapefile(colonies_bbox_psi_popsize, 'colonies_bbox_psi_popsize', bbox_drop_columns)

### Calculate PSI for bbox neighbors using Population Density

In [ ]:
colonies_bbox_psi_popdensity = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       pcen_denom = 'popdensity',
                                       nbr_dist_colname = 'nbrs_dist_bbox')

colonies_bbox_psi_popdensity.head()

export_shapefile(colonies_bbox_psi_popdensity, 'colonies_bbox_psi_popdensity', bbox_drop_columns)

### Calculate PSI for bbox neighbors using Denominator=1

In [ ]:
colonies_bbox_psi_one = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       pcen_denom = 'one',
                                       nbr_dist_colname = 'nbrs_dist_bbox')

export_shapefile(colonies_bbox_psi_one, 'colonies_bbox_psi_one', bbox_drop_columns)

colonies_bbox_psi_one.head()

## Redo PSI Calculations ignoring "Other", "Rural Villages", and areas < .0001

In [ ]:
colonies_pkl_file = 'colonies_bbox_nbrs25Aug2020.pkl'
columns_to_drop = ['nbrs_bbox', 'nbrs_dist_bbox', 'index']
uso_types_to_drop = ['Other', 'RV']
area_cutoff_km2 = .0001
colonies = generate_colonies_with_exclusions(colonies_pkl_file = colonies_pkl_file,
                                             columns_to_drop = columns_to_drop, 
                                             uso_types_to_drop = uso_types_to_drop,
                                             area_cutoff_km2 = area_cutoff_km2)

colonies_bbox_nbrs = colonies.copy()

### Calculate PSI for bbox neighbors using Population Size

In [ ]:
colonies_no_rv_bbox_psi_popsize = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       pcen_denom = 'pop',
                                       nbr_dist_colname = 'nbrs_dist_bbox')

export_shapefile(colonies_no_rv_bbox_psi_popsize, 'colonies_no_rv_bbox_psi_popsize', bbox_drop_columns)

colonies_no_rv_bbox_psi_popsize.head()

### Calculate PSI for bbox neighbors using Population Density

In [ ]:
colonies_no_rv_bbox_psi_popdensity = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       pcen_denom = 'popdensity',
                                       nbr_dist_colname = 'nbrs_dist_bbox')

export_shapefile(colonies_no_rv_bbox_psi_popdensity, 'colonies_no_rv_bbox_psi_popdensity', bbox_drop_columns)

colonies_no_rv_bbox_psi_popdensity.head()

### Calculate PSI for bbox neighbors using Denominator=1

In [ ]:
colonies_no_rv_bbox_psi_one = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       pcen_denom = 'one',
                                       nbr_dist_colname = 'nbrs_dist_bbox')

export_shapefile(colonies_no_rv_bbox_psi_one, 'colonies_no_rv_bbox_psi_one', bbox_drop_columns)

colonies_no_rv_bbox_psi_one.head()